# fewshot_k_feasibility
Token-budget check for K=3/4/5 with FULL exemplars, run on OmniGen2's OWN tokenizer (pipe.processor.tokenizer) and chat template.
Confirms the 1024 limit, demonstrates that exceeding it truncates the real instruction off the end, and reports how many of the 488 prompts overflow at each K.

In [ ]:
from google.colab import drive, userdata
DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Mounted at /content/drive


In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
else:
    print("OmniGen2 already present.")
sys.path.insert(0, REPO); os.chdir(REPO)

In [ ]:
import torch
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
MODEL = "OmniGen2/OmniGen2"
# only the tokenizer is needed here; load the pipeline the same way generation does, then use its tokenizer
pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True)
tok = pipe.processor.tokenizer
print("OmniGen2 tokenizer:", tok.__class__.__name__, "| vocab", len(tok))

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Expected types for transformer: (<class 'omnigen2.models.transformers.transformer_omnigen2.OmniGen2Transformer2DModel'>,), got <class 'diffusers_modules.local.transformer_omnigen2.OmniGen2Transformer2DModel'>.


OmniGen2 tokenizer: Qwen2TokenizerFast | vocab 151669


## The 1024 limit is OmniGen2/OmniContext's operating value

In [ ]:
MAX_SEQ = 1024
SYS = "You are a helpful assistant that generates high-quality images based on user instructions."
def templated_len(prompt):
    text = tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    return len(tok(text).input_ids)                      # untruncated length; add_generation_prompt=True here against False in the pipeline, so this is an upper bound rather than an exact match

print("max_sequence_length we pass in generate():", MAX_SEQ, "(frozen from baseline; not a hard model limit)")
# demonstrate the effect: a prompt over the limit loses its TAIL (where the real instruction sits)
demo = "filler word " * 1200 + "GENERATE THIS EXACT IMAGE: a red laptop on a glass desk."
text = tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":demo}], tokenize=False, add_generation_prompt=True)
full = tok(text).input_ids
trunc = tok(text, max_length=MAX_SEQ, truncation=True).input_ids
print(f"demo prompt: {len(full)} tokens -> pipeline truncates to {len(trunc)} at max_length={MAX_SEQ}")
print("tail after truncation:", repr(tok.decode(trunc[-12:])))
print("=> the real instruction at the end is dropped once the prompt exceeds", MAX_SEQ)

max_sequence_length we pass in generate(): 1024 (frozen from baseline; not a hard model limit)
demo prompt: 2444 tokens -> pipeline truncates to 1024 at max_length=1024
tail after truncation: ' filler word filler word filler word filler word filler word filler word'
=> the real instruction at the end is dropped once the prompt exceeds 1024


## K = 3 / 4 / 5 feasibility with full exemplars

In [ ]:
import json, re, random
from collections import defaultdict
def strip_refs(instr):
    return " ".join(p for p in re.split(r"(?<=\.)\s+", instr) if "<img>" not in p).strip()
train = [json.loads(l) for l in open(f"{DRIVE}/train.jsonl", encoding="utf-8")]
test  = [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
by_t = defaultdict(list)
for r in train: by_t[r["output_image"]].append(strip_refs(r["instruction"]))   # same target -> identical text
targets = list(by_t.keys())
random.seed(42)
assign = {i: [random.choice(by_t[t]) for t in random.sample(targets, 5)] for i in range(len(test))}   # 5 distinct-target exemplars
HEAD = "Here are descriptions of example images:\n"
def build(i, K):
    body = "".join(f"{j+1}. {e}\n" for j, e in enumerate(assign[i][:K]))
    return f"{HEAD}{body}\nNow generate this image: {strip_refs(test[i]['instruction'])}"

import statistics as st
print(f"{'K':>3} | {'mean':>6} {'p95':>6} {'max':>6} | over {MAX_SEQ}")
print("-"*42)
res={}
for K in (3,4,5):
    toks=[templated_len(build(i,K)) for i in range(len(test))]
    over=sum(t>MAX_SEQ for t in toks)
    res[K]=over; print(f"{K:>3} | {st.mean(toks):6.0f} {sorted(toks)[int(0.95*len(toks))]:6.0f} {max(toks):6.0f} | {over:>3} / {len(test)}")
safe=[K for K in (3,4,5) if res[K]==0]
print(f"\nlargest K with ZERO overflow (full exemplars): {max(safe) if safe else 'none'}  -> used for the few-shot arm")

  K |   mean    p95    max | over 1024
------------------------------------------
  3 |    522    581    656 |   0 / 488
  4 |    643    716    793 |   0 / 488
  5 |    764    846    923 |   0 / 488

largest K with ZERO overflow (full exemplars): 5  -> used for the few-shot arm
